### Importar librerias y configuracion general.

In [ ]:
# CELDA: IMPORTACIONES Y CONEXIÓN
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sqlalchemy as sa
import os

# Configuración de Surcomotor S.A.
SERVER = "LAPTOP-OFP910OT"
DATABASE = "Surcomotor_DB"
DIRECT_BASE = "C:/Proyectos/Power BI - Logística/Datos generados/"

# Palabras clave ampliadas para detectar vehículos
MODELOS_VEHICULOS = ["CIVIC", "CR-V", "ACCORD", "PILOT", "FIT", "HR-V", "CITY", "WR-V", "AUTO", "CAMIONETA"]

def categorizar_producto_pro(nombre_prod):
    """Clasificación robusta para Surcomotor S.A."""
    nombre = str(nombre_prod).upper()
    if any(modelo in nombre for modelo in MODELOS_VEHICULOS):
        return "Vehículo"
    return "Repuesto"


### Limpieza de Datos

In [ ]:
# CELDA: FUNCIÓN DE LIMPIEZA MEJORADA
def limpiar_y_preparar(df, nombre_tabla):
    df = df.drop_duplicates()
    
    # Asegurar que la categoría se asigne correctamente antes del EDA
    if 'Producto' in df.columns:
        df['Categoria'] = df['Producto'].apply(categorizar_producto_pro)
    else:
        df['Categoria'] = "General"
        
    # Limpieza de Outliers por Segmento (Z-Score)
    df_limpio = []
    for cat, sub_df in df.groupby('Categoria'):
        sub_df = sub_df.copy()
        for col in sub_df.select_dtypes(include=[np.number]).columns:
            if col in ['id', 'ID_Prod']: continue
            
            # Cálculo de Z-Score: $Z = \frac{x - \mu}{\sigma}$[cite: 1]
            mu, sigma = sub_df[col].mean(), sub_df[col].std()
            if sigma == 0: continue
            
            z_scores = np.abs((sub_df[col] - mu) / sigma)
            
            # Regresión a la media (Z entre 4.5 y 7)
            mask_reg = (z_scores > 4.5) & (z_scores <= 7)
            sub_df.loc[mask_reg, col] = int(round(mu)) if np.issubdtype(sub_df[col].dtype, np.integer) else mu
            
            # Eliminación de errores (Z > 7)
            sub_df = sub_df[z_scores <= 7]
        df_limpio.append(sub_df)
        
    return pd.concat(df_limpio, ignore_index=True)


### Análisis Exploratorio de Datos

In [ ]:
# CELDA: EDA PROFESIONAL CON SEGMENTACIÓN REAL
def eda_final_profesional(df, titulo):
    print(f"\n--- VISUALIZACIÓN ESTRATÉGICA: {titulo} ---")
    
    # Verificar si tenemos ambos segmentos para evitar errores de graficación
    segmentos_presentes = df['Categoria'].unique()
    print(f"Segmentos detectados: {segmentos_presentes}")

    # 1. Boxplot de Precios con Escala Logarítmica (Crucial para Vehículos vs Repuestos)
    col_v = 'Precio_V' if 'Precio_V' in df.columns else ('Precio_C' if 'Precio_C' in df.columns else 'Stock')
    
    plt.figure(figsize=(14, 6))
    sns.boxplot(data=df, x='Categoria', y=col_v, hue='Categoria', palette="Set2", legend=False)
    
    if df[col_v].max() > 1000:
        plt.yscale('log')
        plt.ylabel(f"{col_v} (Escala Logarítmica)")
    
    plt.title(f"Distribución de {col_v} por Unidad de Negocio[cite: 1]")
    plt.show()

    # 2. Análisis de Ingresos por Almacén y Segmento
    if 'Almacen' in df.columns and 'Monto' in df.columns:
        plt.figure(figsize=(14, 6))
        sns.barplot(data=df, x='Almacen', y='Monto', hue='Categoria', estimator=sum, palette="viridis", errorbar=None)
        plt.title("Contribución Financiera por Almacén y Segmento")
        plt.show()


### Ejecución y Carga de Datos a SQL Server

In [ ]:
# CELDA: PROCESAMIENTO Y CARGA
engine = sa.create_engine(f"mssql+pyodbc://@{SERVER}/{DATABASE}?driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes")

for t in ["tabla_ventas", "tabla_compras", "tabla_stock_inventario"]:
    path = os.path.join(DIRECT_BASE, f"{t}.csv")
    if os.path.exists(path):
        df_raw = pd.read_csv(path, encoding='utf-8-sig')
        
        # Limpiar y categorizar
        df_clean = limpiar_y_preparar(df_raw, t)
        
        # Ejecutar EDA
        eda_final_profesional(df_clean, t)
        
        # Carga Segura (Sobrescribe para evitar duplicados acumulados)
        df_clean.to_sql(t, engine, if_exists='replace', index=False)
        print(f"Base de datos actualizada: {t} ({len(df_clean)} registros).")
